# Adrianic v5.4c — Persistent Discourse + Empirical Serial Null

This notebook restores fresh per-edge ref/family permutations and direction XOR, persists discourse context within each chain, locks synthetic marginal head accuracy to the semantic arm, and tests serial error dependence against an empirical independent-error null on the same chains. Direction edge-coherence is intentionally omitted as non-diagnostic.


In [ ]:
# CELL 1 — Install pretrained backend dependencies
!pip -q install "transformers>=4.40"
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


In [ ]:
# CELL 2 — Write the frozen v5.4c harness
from pathlib import Path
script = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nimport re\nimport itertools\nfrom dataclasses import dataclass, asdict\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy.optimize import minimize_scalar\nfrom scipy.stats import wilcoxon\n\n# =============================================================================\n# Adrianic v5.4c — Persistent-Discourse + Serial-Null Semantic Coupling\n#\n# Purpose\n# -------\n# 1) Restore the coupled benchmark\'s observed per-edge constraints: every edge\n#    has a fresh referent permutation, family permutation, and direction XOR.\n# 2) Persist discourse context within each chain: same four names, same scenario,\n#    same context sentence; vary only referring expression and relation template.\n# 3) Separate three emission controls: accuracy only; accuracy + sharpness;\n#    accuracy + sharpness + empirical confusion identity. Test controls are\n#    marginal-accuracy locked to the semantic arm on the evaluated corpus.\n# 4) Score serial error dependence against an empirical independent-error null\n#    on the exact same latent chains/edges. Direction edge-coherence is omitted\n#    because for binary direction it is algebraically pinned by the transition.\n# 5) Keep candidate ranking diagnostic with a shuffled-edge pressure control.\n# 6) Export clauses, persistent discourse IDs, and actual edge constraints.\n#\n# Reproducibility: all RNGs are seed-derived, every run writes config + per-seed\n# summaries + position-level data + clause/edge exports.\n# =============================================================================\n\nRELATIONS = ("contains", "supports", "precedes", "maps")\nDIRECTIONS = ("fwd", "rev")\nN_REF = 3\nN_FAM = len(RELATIONS)\nN_DIR = len(DIRECTIONS)\nN_STATES = N_REF * N_FAM * N_DIR\n\nSCENARIOS = {\n    "carry": {\n        "train_context": ["{a} carried {b} while {c} watched.", "While {c} watched, {a} carried {b}."],\n        "test_context": ["{b} was carried by {a} as {c} watched.", "{c} observed {a} carrying {b}."],\n        "train_ref": {0: ["the carrier", "the one carrying"], 1: ["the carried one", "the one being carried"], 2: ["the watcher", "the one watching"]},\n        "test_ref": {0: ["the person doing the carrying", "the carrier in that scene"], 1: ["the person in transit", "the one transported"], 2: ["the observer", "the witness to the carrying"]},\n    },\n    "handoff": {\n        "train_context": ["{a} handed {b} to {c}.", "{a} passed {b} over to {c}."],\n        "test_context": ["{c} received {b} from {a}.", "From {a}, {c} was given {b}."],\n        "train_ref": {0: ["the giver", "the one handing it over"], 1: ["the transferred one", "the one being handed over"], 2: ["the receiver", "the one receiving it"]},\n        "test_ref": {0: ["the source of the handoff", "the person who gave it"], 1: ["the transferred participant", "the handoff target object"], 2: ["the recipient", "the person who got it"]},\n    },\n    "follow": {\n        "train_context": ["{a} followed {b} while {c} observed.", "With {c} observing, {a} followed {b}."],\n        "test_context": ["{b} was followed by {a} as {c} looked on.", "{c} saw {a} trailing {b}."],\n        "train_ref": {0: ["the follower", "the one following"], 1: ["the one being followed", "the leader"], 2: ["the observer", "the one observing"]},\n        "test_ref": {0: ["the person trailing behind", "the pursuer"], 1: ["the person out front", "the one ahead"], 2: ["the witness", "the person looking on"]},\n    },\n    "escort": {\n        "train_context": ["{a} escorted {b} while {c} waited.", "While {c} waited, {a} escorted {b}."],\n        "test_context": ["{b} was escorted by {a} as {c} waited.", "{c} waited while {a} guided {b}."],\n        "train_ref": {0: ["the escort", "the guide"], 1: ["the escorted one", "the one being guided"], 2: ["the waiter", "the one waiting"]},\n        "test_ref": {0: ["the person doing the guiding", "the guide in that event"], 1: ["the person under escort", "the guided person"], 2: ["the person who stayed behind", "the waiting observer"]},\n    },\n}\n\nREL_TRAIN = {\n    ("contains", "fwd"): ["{ref} contains {d}.", "{ref} has {d} inside."],\n    ("contains", "rev"): ["{ref} is inside {d}.", "{ref} is contained by {d}."],\n    ("supports", "fwd"): ["{ref} supports {d}.", "{ref} holds up {d}."],\n    ("supports", "rev"): ["{ref} is supported by {d}.", "{ref} rests on {d}."],\n    ("precedes", "fwd"): ["{ref} precedes {d}.", "{ref} comes before {d}."],\n    ("precedes", "rev"): ["{ref} follows {d}.", "{ref} comes after {d}."],\n    ("maps", "fwd"): ["{ref} maps to {d}.", "{ref} points to {d}."],\n    ("maps", "rev"): ["{ref} is mapped from {d}.", "{ref} is the target of {d}."],\n}\nREL_TEST = {\n    ("contains", "fwd"): ["Inside {ref} is {d}.", "{d} remains within {ref}."],\n    ("contains", "rev"): ["Inside {d} is {ref}.", "{d} keeps {ref} within."],\n    ("supports", "fwd"): ["Support for {d} comes from {ref}.", "{d} is held up by {ref}."],\n    ("supports", "rev"): ["Support for {ref} comes from {d}.", "{d} holds up {ref}."],\n    ("precedes", "fwd"): ["Before {d} comes {ref}.", "{d} trails behind {ref}."],\n    ("precedes", "rev"): ["After {d} comes {ref}.", "{ref} trails behind {d}."],\n    ("maps", "fwd"): ["To {d}, {ref} points.", "{d} is the target of {ref}."],\n    ("maps", "rev"): ["To {ref}, {d} points.", "{ref} is the target of {d}."],\n}\n\nSYL1 = ("na", "ta", "ve", "ri", "lo", "ke", "mi", "sa", "do", "fi", "ga", "po")\nSYL2 = ("ra", "ven", "ko", "lin", "ma", "tor", "shi", "del", "rin", "ka", "mo", "se")\nTOKEN_RE = re.compile(r"<n\\d>|[a-z]+|[.,]")\n\n\ndef state_id(ref: int, fam: int, direc: int) -> int:\n    return (int(ref) * N_FAM + int(fam)) * N_DIR + int(direc)\n\n\ndef decode_state(s: int) -> Tuple[int, int, int]:\n    s = int(s)\n    direc = s % N_DIR\n    x = s // N_DIR\n    fam = x % N_FAM\n    ref = x // N_FAM\n    return ref, fam, direc\n\n\nSTATE_FACTORS = np.array([decode_state(s) for s in range(N_STATES)], dtype=int)\n\n\ndef unique_names(rng: random.Random, n: int = 4) -> List[str]:\n    out = []\n    while len(out) < n:\n        x = (rng.choice(SYL1) + rng.choice(SYL2)).capitalize()\n        if x not in out:\n            out.append(x)\n    return out\n\n\ndef normalize_names(text: str, names: List[str]) -> str:\n    out = text.lower()\n    for idx, name in sorted(enumerate(names), key=lambda kv: len(kv[1]), reverse=True):\n        out = re.sub(rf"\\b{re.escape(name.lower())}\\b", f"<n{idx}>", out)\n    return out\n\n\ndef sample_discourse_context(rng: random.Random, split: str = "test") -> Dict:\n    """Sample the chain-level context that persists across every clause."""\n    names = unique_names(rng, 4)\n    scenario_key = rng.choice(list(SCENARIOS.keys()))\n    scenario = SCENARIOS[scenario_key]\n    context_key = "train_context" if split == "train" else "test_context"\n    a, b, c, _ = names\n    context_template = rng.choice(scenario[context_key])\n    context = context_template.format(a=a, b=b, c=c)\n    return {\n        "names": names,\n        "scenario_key": scenario_key,\n        "context": context,\n    }\n\n\ndef render_clause(ref: int, fam: int, direc: int, rng: random.Random, split: str = "test",\n                  discourse: Dict | None = None) -> Dict:\n    # Independent rendering remains available for training/vocab construction.\n    if discourse is None:\n        discourse = sample_discourse_context(rng, split=split)\n    names = list(discourse["names"])\n    a, b, c, d = names\n    scenario_key = str(discourse["scenario_key"])\n    scenario = SCENARIOS[scenario_key]\n    ref_key = "train_ref" if split == "train" else "test_ref"\n    rel_table = REL_TRAIN if split == "train" else REL_TEST\n    context = str(discourse["context"])\n    # Only these two linguistic choices vary within a chain.\n    ref_phrase = rng.choice(scenario[ref_key][int(ref)])\n    relation_template = rng.choice(rel_table[(RELATIONS[int(fam)], DIRECTIONS[int(direc)])])\n    target = relation_template.format(ref=ref_phrase, d=d)\n    raw = context + " " + target\n    return {\n        "raw_text": raw,\n        "text": normalize_names(raw, names),\n        "ref": int(ref),\n        "fam": int(fam),\n        "dir": int(direc),\n        "state_id": state_id(ref, fam, direc),\n        "scenario": scenario_key,\n        "context_text": context,\n        "entity_0": names[0],\n        "entity_1": names[1],\n        "entity_2": names[2],\n        "entity_3": names[3],\n    }\n\n\ndef build_vocab() -> Tuple[Dict[str, int], List[str]]:\n    toks = {"<pad>", "<unk>", "<cls>"}\n    rng = random.Random(123)\n    for split in ("train", "test"):\n        for r in range(N_REF):\n            for f in range(N_FAM):\n                for d in range(N_DIR):\n                    for _ in range(6):\n                        row = render_clause(r, f, d, rng, split=split)\n                        toks.update(TOKEN_RE.findall(row["text"].lower()))\n    for i in range(4):\n        toks.add(f"<n{i}>")\n    itos = sorted(toks)\n    return {t: i for i, t in enumerate(itos)}, itos\n\n\nSTOI, ITOS = build_vocab()\nPAD = STOI["<pad>"]\nUNK = STOI["<unk>"]\nCLS = STOI["<cls>"]\n\n\ndef encode(text: str, max_len: int = 48) -> Tuple[List[int], List[int]]:\n    toks = ["<cls>"] + TOKEN_RE.findall(text.lower())\n    ids = [STOI.get(t, UNK) for t in toks[:max_len]]\n    mask = [1] * len(ids)\n    while len(ids) < max_len:\n        ids.append(PAD); mask.append(0)\n    return ids, mask\n\n\nclass FactorTransformer(nn.Module):\n    def __init__(self, d_model: int = 48, nhead: int = 4, layers: int = 1, ff: int = 96, max_len: int = 48):\n        super().__init__()\n        self.emb = nn.Embedding(len(ITOS), d_model, padding_idx=PAD)\n        self.pos = nn.Embedding(max_len, d_model)\n        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=ff, dropout=0.10, batch_first=True, activation="gelu")\n        self.enc = nn.TransformerEncoder(layer, num_layers=layers)\n        self.ref_out = nn.Linear(d_model, N_REF)\n        self.fam_out = nn.Linear(d_model, N_FAM)\n        self.dir_out = nn.Linear(d_model, N_DIR)\n\n    def forward(self, ids, mask):\n        b, t = ids.shape\n        pos = torch.arange(t, device=ids.device)[None, :].expand(b, t)\n        z = self.enc(self.emb(ids) + self.pos(pos), src_key_padding_mask=(mask == 0))\n        h = z[:, 0, :]\n        return self.ref_out(h), self.fam_out(h), self.dir_out(h)\n\n\n@dataclass\nclass Config:\n    seeds: int = 8\n    chains_per_seed: int = 8\n    chain_len: int = 80\n    train_repeats_per_state: int = 14\n    transformer_epochs: int = 8\n    label_noise: float = 0.04\n    semantic_backend: str = "tiny"\n    pretrained_model: str = "distilbert-base-uncased"\n    pretrained_epochs: int = 4\n    pretrained_batch_size: int = 16\n    pretrained_lr: float = 2e-5\n    # Coupled-benchmark edge reliability. Each edge has fresh permutations/XOR.\n    edge_ref_reliability: float = 0.78\n    edge_fam_reliability: float = 0.75\n    edge_dir_reliability: float = 0.78\n    max_k: int = 12\n    min_k: int = 2\n    null_simulations: int = 500\n    output_dir: str = "/mnt/data/adrianic_v54c_persistent_discourse_results"\n\n\ndef _factor_transition(n: int, perm: np.ndarray, reliability: float) -> np.ndarray:\n    if not (0.0 < reliability < 1.0):\n        raise ValueError("edge reliability must lie in (0,1)")\n    out = np.full((n, n), (1.0 - reliability) / (n - 1), float)\n    for prev in range(n):\n        out[prev, int(perm[prev])] = reliability\n    return out\n\n\ndef edge_log_matrix(ref_perm, fam_perm, dir_xor: int, cfg: Config) -> np.ndarray:\n    ref_perm = np.asarray(ref_perm, int)\n    fam_perm = np.asarray(fam_perm, int)\n    if sorted(ref_perm.tolist()) != list(range(N_REF)):\n        raise ValueError("ref_perm is not a permutation")\n    if sorted(fam_perm.tolist()) != list(range(N_FAM)):\n        raise ValueError("fam_perm is not a permutation")\n    if int(dir_xor) not in (0, 1):\n        raise ValueError("dir_xor must be 0/1")\n    Tr = _factor_transition(N_REF, ref_perm, cfg.edge_ref_reliability)\n    Tf = _factor_transition(N_FAM, fam_perm, cfg.edge_fam_reliability)\n    Td = np.full((N_DIR, N_DIR), 1.0 - cfg.edge_dir_reliability, float)\n    for prev in range(N_DIR):\n        Td[prev, prev ^ int(dir_xor)] = cfg.edge_dir_reliability\n    r, f, d = STATE_FACTORS.T\n    T = (Tr[r[:, None], r[None, :]] *\n         Tf[f[:, None], f[None, :]] *\n         Td[d[:, None], d[None, :]])\n    assert np.allclose(T.sum(axis=1), 1.0)\n    return np.log(T + 1e-300)\n\n\ndef sample_chain(rng: np.random.Generator, length: int, cfg: Config, chain_id: str):\n    states = np.empty(length, dtype=int)\n    states[0] = int(rng.integers(N_STATES))\n    edge_logs, edge_meta = [], []\n    for t in range(length - 1):\n        ref_perm = rng.permutation(N_REF).astype(int)\n        fam_perm = rng.permutation(N_FAM).astype(int)\n        dir_xor = int(rng.integers(2))\n        logT = edge_log_matrix(ref_perm, fam_perm, dir_xor, cfg)\n        Trow = np.exp(logT[states[t]])\n        states[t + 1] = int(rng.choice(N_STATES, p=Trow))\n        edge_logs.append(logT)\n        edge_meta.append({\n            "chain_id": chain_id,\n            "src_position": int(t),\n            "dst_position": int(t + 1),\n            "ref_perm": ref_perm.tolist(),\n            "fam_perm": fam_perm.tolist(),\n            "dir_xor": dir_xor,\n        })\n    return states, edge_logs, edge_meta\n\n\ndef make_chain_dataframe(seed: int, cfg: Config, split: str = "test", tag: str = "test"):\n    nrng = np.random.default_rng(seed + 9000)\n    rrng = random.Random(seed + 19000)\n    rows, edge_rows = [], []\n    edge_tables = {}\n    for c in range(cfg.chains_per_seed):\n        cid = f"{tag}_s{seed}_c{c}"\n        chain, logs, meta = sample_chain(nrng, cfg.chain_len, cfg, cid)\n        edge_tables[cid] = logs\n        clause_ids = [f"{cid}_p{p}" for p in range(cfg.chain_len)]\n        discourse = sample_discourse_context(rrng, split=split)\n        for pos, s in enumerate(chain):\n            r, f, d = decode_state(int(s))\n            x = render_clause(r, f, d, rrng, split=split, discourse=discourse)\n            x.update({"seed": seed, "chain_id": cid, "position": pos, "clause_id": clause_ids[pos],\n                      "discourse_id": cid})\n            rows.append(x)\n        for e in meta:\n            e = dict(e)\n            e.update({\n                "seed": seed,\n                "src_clause_id": clause_ids[e["src_position"]],\n                "dst_clause_id": clause_ids[e["dst_position"]],\n                "edge_type": "observed_permutation_xor",\n                "edge_ref_reliability": cfg.edge_ref_reliability,\n                "edge_fam_reliability": cfg.edge_fam_reliability,\n                "edge_dir_reliability": cfg.edge_dir_reliability,\n                "ref_perm": json.dumps(e["ref_perm"]),\n                "fam_perm": json.dumps(e["fam_perm"]),\n            })\n            edge_rows.append(e)\n    df = pd.DataFrame(rows)\n    # Persistent-discourse invariant: every chain keeps exactly one scenario,\n    # context sentence, and four entity identities.\n    for cid, g in df.groupby("chain_id"):\n        for col in ("scenario","context_text","entity_0","entity_1","entity_2","entity_3"):\n            if g[col].nunique(dropna=False) != 1:\n                raise AssertionError(f"persistent discourse invariant failed for {cid} column {col}")\n    return df, edge_tables, pd.DataFrame(edge_rows)\n\n\ndef shuffled_edge_tables(edge_tables: Dict[str, List[np.ndarray]], seed: int) -> Dict[str, List[np.ndarray]]:\n    """Pressure-preserving null: same edge matrices, assigned to the wrong adjacencies."""\n    rng = np.random.default_rng(seed + 47000)\n    flat = [m for cid in edge_tables for m in edge_tables[cid]]\n    order = rng.permutation(len(flat))\n    shuffled = [flat[int(i)] for i in order]\n    out, k = {}, 0\n    for cid, mats in edge_tables.items():\n        out[cid] = shuffled[k:k + len(mats)]\n        k += len(mats)\n    return out\n\n\ndef make_train_dataframe(seed: int, cfg: Config) -> pd.DataFrame:\n    rrng = random.Random(seed + 29000)\n    rows = []\n    for s in range(N_STATES):\n        r, f, d = decode_state(s)\n        for _ in range(cfg.train_repeats_per_state):\n            rows.append(render_clause(r, f, d, rrng, split="train"))\n    rrng.shuffle(rows)\n    return pd.DataFrame(rows)\n\n\ndef make_calibration_dataframe(seed: int, cfg: Config):\n    # Independent chain-distributed calibration set: same held-out renderer, fresh edges/text.\n    return make_chain_dataframe(seed + 700_000, cfg, split="test", tag="calib")\n\n\ndef tensorize(df: pd.DataFrame):\n    ids, masks = zip(*(encode(x) for x in df.text.tolist()))\n    return (\n        torch.tensor(ids, dtype=torch.long),\n        torch.tensor(masks, dtype=torch.long),\n        torch.tensor(df.ref.to_numpy(), dtype=torch.long),\n        torch.tensor(df.fam.to_numpy(), dtype=torch.long),\n        torch.tensor(df.dir.to_numpy(), dtype=torch.long),\n    )\n\n\ndef train_transformer(seed: int, train_df: pd.DataFrame, cfg: Config) -> FactorTransformer:\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    model = FactorTransformer()\n    opt = torch.optim.AdamW(model.parameters(), lr=0.0035, weight_decay=1e-4)\n    ids, mask, yr, yf, yd = tensorize(train_df)\n    n = len(train_df)\n    for _ in range(cfg.transformer_epochs):\n        order = torch.randperm(n)\n        for start in range(0, n, 64):\n            idx = order[start:start+64]\n            rr, ff, dd = model(ids[idx], mask[idx])\n            ar, af, ad = yr[idx].clone(), yf[idx].clone(), yd[idx].clone()\n            if cfg.label_noise > 0:\n                nm = torch.rand(len(idx)) < cfg.label_noise\n                if int(nm.sum()):\n                    ar[nm] = torch.randint(0, N_REF, (int(nm.sum()),))\n                    af[nm] = torch.randint(0, N_FAM, (int(nm.sum()),))\n                    ad[nm] = torch.randint(0, N_DIR, (int(nm.sum()),))\n            loss = F.cross_entropy(rr, ar) + F.cross_entropy(ff, af) + F.cross_entropy(dd, ad)\n            opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()\n    return model\n\n\ndef transformer_logits(model: FactorTransformer, df: pd.DataFrame) -> Dict[str, np.ndarray]:\n    ids, mask, *_ = tensorize(df)\n    out = {"ref": [], "fam": [], "dir": []}\n    model.eval()\n    with torch.no_grad():\n        for start in range(0, len(df), 128):\n            rr, ff, dd = model(ids[start:start+128], mask[start:start+128])\n            out["ref"].append(rr.numpy()); out["fam"].append(ff.numpy()); out["dir"].append(dd.numpy())\n    return {k: np.concatenate(v, axis=0) for k, v in out.items()}\n\n\n\ndef require_transformers():\n    try:\n        from transformers import AutoTokenizer, AutoModel\n        return AutoTokenizer, AutoModel\n    except ImportError as e:\n        raise RuntimeError(\n            "Pretrained backend requires transformers. In Colab run: "\n            "!pip -q install transformers sentencepiece"\n        ) from e\n\n\nclass PretrainedFactorHeads(nn.Module):\n    def __init__(self, model_name: str):\n        super().__init__()\n        _, AutoModel = require_transformers()\n        self.encoder = AutoModel.from_pretrained(model_name)\n        hidden = int(self.encoder.config.hidden_size)\n        self.ref_out = nn.Linear(hidden, N_REF)\n        self.fam_out = nn.Linear(hidden, N_FAM)\n        self.dir_out = nn.Linear(hidden, N_DIR)\n\n    def forward(self, **batch):\n        out = self.encoder(**batch)\n        h = out.last_hidden_state[:, 0, :]\n        return self.ref_out(h), self.fam_out(h), self.dir_out(h)\n\n\ndef train_pretrained(seed: int, train_df: pd.DataFrame, cfg: Config):\n    AutoTokenizer, _ = require_transformers()\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    tokenizer = AutoTokenizer.from_pretrained(cfg.pretrained_model)\n    model = PretrainedFactorHeads(cfg.pretrained_model).to(device)\n    enc = tokenizer(\n        train_df.raw_text.tolist(), padding=True, truncation=True,\n        max_length=96, return_tensors="pt"\n    )\n    yr = torch.tensor(train_df.ref.to_numpy(), dtype=torch.long)\n    yf = torch.tensor(train_df.fam.to_numpy(), dtype=torch.long)\n    yd = torch.tensor(train_df.dir.to_numpy(), dtype=torch.long)\n    opt = torch.optim.AdamW(model.parameters(), lr=cfg.pretrained_lr, weight_decay=1e-4)\n    n = len(train_df)\n    for epoch in range(cfg.pretrained_epochs):\n        order = torch.randperm(n)\n        model.train(); running = 0.0\n        for start in range(0, n, cfg.pretrained_batch_size):\n            idx = order[start:start+cfg.pretrained_batch_size]\n            batch = {k: v[idx].to(device) for k,v in enc.items()}\n            rr, ff, dd = model(**batch)\n            loss = (F.cross_entropy(rr, yr[idx].to(device)) +\n                    F.cross_entropy(ff, yf[idx].to(device)) +\n                    F.cross_entropy(dd, yd[idx].to(device)))\n            opt.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()\n            running += float(loss.item()) * len(idx)\n        print(f"    pretrained epoch {epoch+1}/{cfg.pretrained_epochs} loss={running/n:.4f}", flush=True)\n    return tokenizer, model\n\n\ndef pretrained_logits(tokenizer, model: PretrainedFactorHeads, df: pd.DataFrame) -> Dict[str, np.ndarray]:\n    device = next(model.parameters()).device\n    enc = tokenizer(df.raw_text.tolist(), padding=True, truncation=True, max_length=96, return_tensors="pt")\n    out = {"ref": [], "fam": [], "dir": []}\n    model.eval()\n    with torch.no_grad():\n        for start in range(0, len(df), 64):\n            sl = slice(start, min(len(df), start+64))\n            batch = {k: v[sl].to(device) for k,v in enc.items()}\n            rr, ff, dd = model(**batch)\n            out["ref"].append(rr.cpu().numpy()); out["fam"].append(ff.cpu().numpy()); out["dir"].append(dd.cpu().numpy())\n    return {k: np.concatenate(v, axis=0) for k,v in out.items()}\n\n\ndef acc_from_logits(logits: np.ndarray, y: np.ndarray) -> float:\n    return float(np.mean(np.argmax(logits, axis=1) == y))\n\n\ndef tune_signal_to_accuracy(y: np.ndarray, noise: np.ndarray, target_acc: float) -> float:\n    def acc(sig):\n        z = noise.copy(); z[np.arange(len(y)), y] += sig\n        return np.mean(np.argmax(z, axis=1) == y)\n    lo, hi = 0.0, 12.0\n    for _ in range(50):\n        mid = (lo + hi) / 2\n        if acc(mid) < target_acc: lo = mid\n        else: hi = mid\n    return (lo + hi) / 2\n\n\ndef empirical_confusion(logits: np.ndarray, y: np.ndarray, k: int) -> np.ndarray:\n    pred = np.argmax(logits, axis=1)\n    C = np.zeros((k, k), float)\n    for true in range(k):\n        idx = np.flatnonzero(y == true)\n        if not len(idx):\n            C[true] = 1.0 / k\n            continue\n        counts = np.bincount(pred[idx], minlength=k).astype(float)\n        C[true] = counts / counts.sum()\n    return C\n\n\ndef _tune_class_signal(noise: np.ndarray, true_class: int, target_acc: float) -> float:\n    target_acc = float(np.clip(target_acc, 0.0, 1.0))\n    # Allow negative signal so a control can match semantic per-class accuracy\n    # below the uninformed Gaussian baseline. Accuracy is monotone in this shift.\n    lo, hi = -12.0, 12.0\n    for _ in range(60):\n        mid = (lo + hi) / 2\n        z = noise.copy(); z[:, true_class] += mid\n        a = float(np.mean(np.argmax(z, axis=1) == true_class))\n        if a < target_acc:\n            lo = mid\n        else:\n            hi = mid\n    # Finite samples make the objective stepwise; choose the closer endpoint.\n    candidates = (lo, (lo+hi)/2, hi)\n    best = None\n    for sig in candidates:\n        z = noise.copy(); z[:, true_class] += sig\n        a = float(np.mean(np.argmax(z, axis=1) == true_class))\n        key = (abs(a-target_acc), abs(sig))\n        if best is None or key < best[0]: best = (key, sig)\n    return float(best[1])\n\n\ndef make_accuracy_controls(seed: int, calib_df: pd.DataFrame, test_df: pd.DataFrame,\n                           sem_calib: Dict[str, np.ndarray], sem_test: Dict[str, np.ndarray]):\n    """Independent Gaussian controls. Calibration controls match calibration\n    semantic accuracy; evaluated test controls are separately locked to the\n    semantic arm\'s per-class marginal accuracy on the current test corpus.\n    This uses semantic outcomes only to set one-site marginals, never row-level\n    correctness or serial placement."""\n    ys_cal = {"ref": calib_df.ref.to_numpy(int), "fam": calib_df.fam.to_numpy(int), "dir": calib_df.dir.to_numpy(int)}\n    ys_test = {"ref": test_df.ref.to_numpy(int), "fam": test_df.fam.to_numpy(int), "dir": test_df.dir.to_numpy(int)}\n    sizes = {"ref": N_REF, "fam": N_FAM, "dir": N_DIR}\n    rng = np.random.default_rng(seed + 39000)\n    cal_out, test_out, signals, confusions_cal, confusions_test = {}, {}, {}, {}, {}\n    for h in ("ref", "fam", "dir"):\n        K = sizes[h]\n        Ccal = empirical_confusion(sem_calib[h], ys_cal[h], K)\n        Ctest = empirical_confusion(sem_test[h], ys_test[h], K)\n        confusions_cal[h] = Ccal; confusions_test[h] = Ctest\n        ncal = rng.normal(size=(len(calib_df), K))\n        ntest = rng.normal(size=(len(test_df), K))\n        zcal = ncal.copy(); ztest = ntest.copy()\n        sig_cal = np.zeros(K, float); sig_test = np.zeros(K, float)\n        for cls in range(K):\n            idx = np.flatnonzero(ys_cal[h] == cls)\n            sigc = _tune_class_signal(ncal[idx], cls, Ccal[cls, cls]) if len(idx) else 0.0\n            sig_cal[cls] = sigc\n            zcal[idx, cls] += sigc\n            idt = np.flatnonzero(ys_test[h] == cls)\n            sigt = _tune_class_signal(ntest[idt], cls, Ctest[cls, cls]) if len(idt) else 0.0\n            sig_test[cls] = sigt\n            ztest[idt, cls] += sigt\n        cal_out[h] = zcal; test_out[h] = ztest\n        signals[h] = {"calibration": sig_cal, "test": sig_test}\n        # Assert the evaluated control matches semantic per-class accuracy to\n        # finite-sample resolution (ties in the signal search can differ by 1 row).\n        for cls in range(K):\n            idt = np.flatnonzero(ys_test[h] == cls)\n            if len(idt):\n                a_sem = np.mean(np.argmax(sem_test[h][idt], axis=1) == cls)\n                a_syn = np.mean(np.argmax(ztest[idt], axis=1) == cls)\n                if abs(a_sem - a_syn) > (1.01 / len(idt)):\n                    raise AssertionError(f"{h} class {cls} marginal accuracy lock failed: {a_syn} vs {a_sem}")\n    return cal_out, test_out, signals, confusions_cal, confusions_test\n\n\ndef impose_confusion_identity(logits: np.ndarray, y: np.ndarray, C: np.ndarray,\n                              seed: int) -> np.ndarray:\n    """Change only WHICH wrong class wins. Preserve correctness, true logit,\n    true margin, centered strength, and the full row-wise multiset of logits."""\n    rng = np.random.default_rng(seed)\n    z = logits.copy()\n    before_correct = np.argmax(z, axis=1) == y\n    before_true = z[np.arange(len(y)), y].copy()\n    before_sorted = np.sort(z, axis=1).copy()\n    K = z.shape[1]\n    for true in range(K):\n        idx = np.flatnonzero((y == true) & (~before_correct))\n        if not len(idx):\n            continue\n        probs = C[true].copy(); probs[true] = 0.0\n        if probs.sum() <= 0:\n            probs[:] = 1.0; probs[true] = 0.0\n        probs /= probs.sum()\n        raw = probs * len(idx)\n        counts = np.floor(raw).astype(int)\n        rem = len(idx) - int(counts.sum())\n        frac_order = np.argsort(-(raw - counts))\n        for j in frac_order:\n            if rem <= 0: break\n            if j == true: continue\n            counts[j] += 1; rem -= 1\n        targets = np.concatenate([np.full(counts[j], j, int) for j in range(K) if j != true])\n        rng.shuffle(targets)\n        if len(targets) != len(idx):\n            raise AssertionError("confusion allocation size mismatch")\n        for ii, target in zip(idx, targets):\n            cur = int(np.argmax(z[ii]))\n            target = int(target)\n            if cur != target:\n                z[ii, cur], z[ii, target] = z[ii, target], z[ii, cur]\n    after_correct = np.argmax(z, axis=1) == y\n    assert np.array_equal(before_correct, after_correct), "confusion remap changed row correctness"\n    assert np.allclose(before_true, z[np.arange(len(y)), y]), "confusion remap changed true logits"\n    assert np.allclose(before_sorted, np.sort(z, axis=1)), "confusion remap changed logit spectrum"\n    return z\n\n\ndef log_softmax_np(z: np.ndarray) -> np.ndarray:\n    z = z - z.max(axis=1, keepdims=True)\n    return z - np.log(np.exp(z).sum(axis=1, keepdims=True) + 1e-300)\n\n\ndef centered_strength(z: np.ndarray) -> np.ndarray:\n    c = z - z.mean(axis=1, keepdims=True)\n    return np.linalg.norm(c, axis=1)\n\n\ndef true_margin(z: np.ndarray, y: np.ndarray) -> np.ndarray:\n    zy = z[np.arange(len(y)), y]\n    tmp = z.copy(); tmp[np.arange(len(y)), y] = -np.inf\n    return zy - tmp.max(axis=1)\n\n\ndef fit_temperature(source_logits: np.ndarray, target_logits: np.ndarray, y: np.ndarray) -> float:\n    src_correct = np.argmax(source_logits, axis=1) == y\n    tgt_correct = np.argmax(target_logits, axis=1) == y\n    qs = np.array([0.1, 0.25, 0.5, 0.75, 0.9])\n\n    def features(z, mask):\n        if mask.sum() < 8:\n            mask = np.ones(len(z), dtype=bool)\n        s = centered_strength(z[mask]); m = true_margin(z[mask], y[mask])\n        return np.r_[np.quantile(s, qs), np.quantile(m, qs)]\n\n    target_vec = np.r_[features(target_logits, tgt_correct), features(target_logits, ~tgt_correct)]\n    scale = np.maximum(np.abs(target_vec), 0.25)\n\n    def objective(logT):\n        T = math.exp(logT)\n        z = source_logits / T\n        src_vec = np.r_[features(z, src_correct), features(z, ~src_correct)]\n        return float(np.mean(((src_vec - target_vec) / scale) ** 2))\n\n    res = minimize_scalar(objective, bounds=(math.log(0.25), math.log(8.0)), method="bounded")\n    return float(math.exp(res.x))\n\n\ndef joint_log_emission(head_logits: Dict[str, np.ndarray]) -> np.ndarray:\n    lr = log_softmax_np(head_logits["ref"])\n    lf = log_softmax_np(head_logits["fam"])\n    ld = log_softmax_np(head_logits["dir"])\n    out = np.empty((len(lr), N_STATES), float)\n    for s, (r, f, d) in enumerate(STATE_FACTORS):\n        out[:, s] = lr[:, r] + lf[:, f] + ld[:, d]\n    return out\n\n\ndef forward_backward(log_emission: np.ndarray, edge_tables: Dict[str, List[np.ndarray]], chain_ids: np.ndarray) -> np.ndarray:\n    post = np.empty_like(log_emission)\n    for cid in dict.fromkeys(chain_ids.tolist()):\n        idx = np.flatnonzero(chain_ids == cid)\n        E = log_emission[idx]\n        edges = edge_tables[str(cid)]\n        L = len(idx)\n        if len(edges) != L - 1:\n            raise AssertionError("edge count does not match chain length")\n        a = np.empty_like(E); b = np.empty_like(E)\n        a[0] = E[0] - math.log(N_STATES)\n        for t in range(1, L):\n            m = a[t-1][:, None] + edges[t-1]\n            mx = m.max(axis=0)\n            a[t] = E[t] + mx + np.log(np.exp(m - mx).sum(axis=0) + 1e-300)\n        b[-1] = 0.0\n        for t in range(L-2, -1, -1):\n            m = edges[t] + E[t+1][None, :] + b[t+1][None, :]\n            mx = m.max(axis=1)\n            b[t] = mx + np.log(np.exp(m - mx[:, None]).sum(axis=1) + 1e-300)\n        q = a + b\n        q -= q.max(axis=1, keepdims=True)\n        q -= np.log(np.exp(q).sum(axis=1, keepdims=True) + 1e-300)\n        post[idx] = q\n    return post\n\n\ndef one_pass_coupling_scores(log_emission: np.ndarray, edge_tables: Dict[str, List[np.ndarray]], chain_ids: np.ndarray) -> np.ndarray:\n    # One local-neighbor sum-product pass. No candidate set enters the score.\n    p = np.exp(log_emission - log_emission.max(axis=1, keepdims=True))\n    p /= p.sum(axis=1, keepdims=True)\n    scores = log_emission.copy()\n    for cid in dict.fromkeys(chain_ids.tolist()):\n        idx = np.flatnonzero(chain_ids == cid)\n        edges = edge_tables[str(cid)]\n        for j, ii in enumerate(idx):\n            msg = np.zeros(N_STATES); count = 0\n            if j > 0:\n                T = np.exp(edges[j-1])\n                msg += np.log(p[idx[j-1]] @ T + 1e-300); count += 1\n            if j < len(idx)-1:\n                T = np.exp(edges[j])\n                msg += np.log(T @ p[idx[j+1]] + 1e-300); count += 1\n            if count:\n                scores[ii] += msg / count\n    return scores\n\n\ndef entropy_soft_k(log_scores: np.ndarray, min_k: int, max_k: int) -> np.ndarray:\n    z = log_scores - log_scores.max(axis=1, keepdims=True)\n    p = np.exp(z); p /= p.sum(axis=1, keepdims=True)\n    H = -(p * np.log(p + 1e-300)).sum(axis=1)\n    k = np.ceil(np.exp(H)).astype(int)\n    return np.clip(k, min_k, max_k)\n\n\ndef topk_mask(scores: np.ndarray, ks: np.ndarray) -> np.ndarray:\n    mask = np.zeros_like(scores, dtype=bool)\n    order = np.argsort(-scores, axis=1)\n    for i, k in enumerate(ks): mask[i, order[i, :int(k)]] = True\n    return mask\n\n\ndef masked_forward_backward(log_emission: np.ndarray, edge_tables: Dict[str, List[np.ndarray]], chain_ids: np.ndarray, mask: np.ndarray) -> np.ndarray:\n    E = log_emission.copy(); E[~mask] = -1e12\n    return forward_backward(E, edge_tables, chain_ids)\n\n\ndef factor_metrics(logits: Dict[str, np.ndarray], df: pd.DataFrame) -> Dict[str, float]:\n    ys = {"ref": df.ref.to_numpy(int), "fam": df.fam.to_numpy(int), "dir": df.dir.to_numpy(int)}\n    out = {}\n    for k in ys:\n        z = logits[k]; y = ys[k]\n        out[f"{k}_acc"] = acc_from_logits(z, y)\n        out[f"{k}_strength"] = float(centered_strength(z).mean())\n        out[f"{k}_margin"] = float(true_margin(z, y).mean())\n    return out\n\n\ndef exact_state_accuracy(log_scores: np.ndarray, y: np.ndarray) -> float:\n    return float(np.mean(np.argmax(log_scores, axis=1) == y))\n\n\ndef candidate_diagnostic(localE, exact_post, rank_scores, shuffled_scores, df, edge_tables, cfg):\n    y = df.state_id.to_numpy(int)\n    chains = df.chain_id.to_numpy()\n    ks = entropy_soft_k(localE, cfg.min_k, cfg.max_k)\n    local_mask = topk_mask(localE, ks)\n    coupling_mask = topk_mask(rank_scores, ks)\n    shuffled_mask = topk_mask(shuffled_scores, ks)\n    local_cov = local_mask[np.arange(len(y)), y]\n    coupling_cov = coupling_mask[np.arange(len(y)), y]\n    shuffled_cov = shuffled_mask[np.arange(len(y)), y]\n    rescued = coupling_cov & (~local_cov)\n\n    soft_post = masked_forward_backward(localE, edge_tables, chains, coupling_mask)\n    cond = exact_post.copy(); cond[~coupling_mask] = -1e12\n\n    def acc_on(scores, mask):\n        return float(np.mean(np.argmax(scores[mask], axis=1) == y[mask])) if mask.any() else float(\'nan\')\n\n    return {\n        "mean_k": float(ks.mean()),\n        "local_coverage": float(local_cov.mean()),\n        "coupling_coverage": float(coupling_cov.mean()),\n        "shuffled_coverage": float(shuffled_cov.mean()),\n        "coupling_minus_local_pp": float(100*(coupling_cov.mean()-local_cov.mean())),\n        "coupling_minus_shuffled_pp": float(100*(coupling_cov.mean()-shuffled_cov.mean())),\n        "rescued_n": int(rescued.sum()),\n        "rescued_fraction": float(rescued.mean()),\n        "rescued_local_argmax": acc_on(localE, rescued),\n        "rescued_coupling_softwall": acc_on(soft_post, rescued),\n        "rescued_structured_exact": acc_on(exact_post, rescued),\n        "rescued_structured_exact_candidate_conditioned": acc_on(cond, rescued),\n    }\n\n\ndef _prepare_serial_index(df: pd.DataFrame, edges_df: pd.DataFrame) -> Dict:\n    row_by_clause = {cid: i for i, cid in enumerate(df.clause_id.tolist())}\n    src = np.array([row_by_clause[x] for x in edges_df.src_clause_id.tolist()], dtype=int)\n    dst = np.array([row_by_clause[x] for x in edges_df.dst_clause_id.tolist()], dtype=int)\n    ref_perm = np.array([json.loads(x) for x in edges_df.ref_perm.tolist()], dtype=int)\n    fam_perm = np.array([json.loads(x) for x in edges_df.fam_perm.tolist()], dtype=int)\n    chain_names = list(dict.fromkeys(df.chain_id.tolist()))\n    chain_code_map = {c:i for i,c in enumerate(chain_names)}\n    chain_codes = np.array([chain_code_map[c] for c in df.chain_id.tolist()], dtype=int)\n    chain_counts = np.bincount(chain_codes, minlength=len(chain_names)).astype(float)\n    return {"src":src, "dst":dst, "ref_perm":ref_perm, "fam_perm":fam_perm,\n            "chain_codes":chain_codes, "chain_counts":chain_counts, "n_chains":len(chain_names)}\n\n\ndef _serial_metrics_from_preds(preds: Dict[str, np.ndarray], df: pd.DataFrame, edges_df: pd.DataFrame,\n                               prepared: Dict | None = None) -> Dict[str, float]:\n    truths = {"ref": df.ref.to_numpy(int), "fam": df.fam.to_numpy(int), "dir": df.dir.to_numpy(int)}\n    q = _prepare_serial_index(df, edges_df) if prepared is None else prepared\n    src, dst = q["src"], q["dst"]\n    out = {}\n    for h in ("ref", "fam", "dir"):\n        err = (preds[h] != truths[h]).astype(float)\n        e0, e1 = err[src], err[dst]\n        out[f"{h}_error_lag1_corr"] = float(np.corrcoef(e0, e1)[0,1]) if e0.std() > 0 and e1.std() > 0 else float("nan")\n        sums = np.bincount(q["chain_codes"], weights=err, minlength=q["n_chains"])\n        rates = sums / q["chain_counts"]\n        out[f"{h}_chain_error_rate_var"] = float(np.var(rates, ddof=1)) if len(rates) > 1 else float("nan")\n        if h in ("ref", "fam"):\n            both = (e0 > 0.5) & (e1 > 0.5)\n            if np.any(both):\n                perm = q["ref_perm"] if h == "ref" else q["fam_perm"]\n                expected = perm[np.arange(len(src)), preds[h][src]]\n                coherent = preds[h][dst] == expected\n                out[f"{h}_both_wrong_edge_coherence"] = float(np.mean(coherent[both]))\n                out[f"{h}_both_wrong_edges"] = int(np.sum(both))\n            else:\n                out[f"{h}_both_wrong_edge_coherence"] = float("nan")\n                out[f"{h}_both_wrong_edges"] = 0\n    return out\n\n\ndef _sample_independent_preds(y: np.ndarray, C: np.ndarray, rng: np.random.Generator) -> np.ndarray:\n    p = np.empty(len(y), dtype=int)\n    for cls in range(C.shape[0]):\n        idx = np.flatnonzero(y == cls)\n        if len(idx):\n            probs = np.asarray(C[cls], float)\n            probs = probs / probs.sum()\n            p[idx] = rng.choice(C.shape[1], size=len(idx), p=probs)\n    return p\n\n\ndef serial_error_against_independent_null(logits: Dict[str, np.ndarray], df: pd.DataFrame, edges_df: pd.DataFrame,\n                                          seed: int, simulations: int = 500) -> Dict[str, float]:\n    """Compare observed serial diagnostics with independent errors sampled from\n    the arm\'s own empirical one-site confusion matrix on the exact same chains."""\n    ys = {"ref": df.ref.to_numpy(int), "fam": df.fam.to_numpy(int), "dir": df.dir.to_numpy(int)}\n    sizes = {"ref": N_REF, "fam": N_FAM, "dir": N_DIR}\n    obs_preds = {h: np.argmax(logits[h], axis=1) for h in ("ref","fam","dir")}\n    prepared = _prepare_serial_index(df, edges_df)\n    observed = _serial_metrics_from_preds(obs_preds, df, edges_df, prepared=prepared)\n    Cs = {h: empirical_confusion(logits[h], ys[h], sizes[h]) for h in sizes}\n    metric_names = []\n    for h in ("ref","fam","dir"):\n        metric_names += [f"{h}_error_lag1_corr", f"{h}_chain_error_rate_var"]\n        if h in ("ref","fam"):\n            metric_names.append(f"{h}_both_wrong_edge_coherence")\n    sims = {k: [] for k in metric_names}\n    rng = np.random.default_rng(seed)\n    for _ in range(int(simulations)):\n        pp = {h: _sample_independent_preds(ys[h], Cs[h], rng) for h in ("ref","fam","dir")}\n        m = _serial_metrics_from_preds(pp, df, edges_df, prepared=prepared)\n        for k in metric_names:\n            if np.isfinite(m[k]): sims[k].append(float(m[k]))\n    out = dict(observed)\n    for k in metric_names:\n        arr = np.asarray(sims[k], float)\n        mu = float(arr.mean()) if len(arr) else float("nan")\n        sd = float(arr.std(ddof=1)) if len(arr) > 1 else float("nan")\n        obs = float(observed[k])\n        z = (obs-mu)/sd if np.isfinite(obs) and np.isfinite(sd) and sd > 0 else float("nan")\n        p = ((1 + np.sum(np.abs(arr-mu) >= abs(obs-mu))) / (len(arr)+1)) if len(arr) and np.isfinite(obs) else float("nan")\n        out[f"{k}_null_mean"] = mu\n        out[f"{k}_null_sd"] = sd\n        out[f"{k}_excess"] = obs - mu if np.isfinite(obs) and np.isfinite(mu) else float("nan")\n        out[f"{k}_z"] = float(z)\n        out[f"{k}_p_two_sided"] = float(p)\n    out["direction_edge_coherence_omitted"] = 1.0\n    return out\n\n\ndef self_test():\n    cfg = Config(seeds=1, chains_per_seed=1, chain_len=3)\n    ident_r = np.arange(N_REF); ident_f = np.arange(N_FAM)\n    logT = edge_log_matrix(ident_r, ident_f, 0, cfg)\n    edge_tables = {"x": [logT, logT]}\n    E = np.full((3, N_STATES), -math.log(N_STATES))\n    target = state_id(1, 2, 0)\n    E[0, target] += 5.0; E[2, target] += 5.0\n    chains = np.array(["x", "x", "x"])\n    base_margin = E[1, target] - np.max(np.delete(E[1], target))\n    scores = one_pass_coupling_scores(E, edge_tables, chains)\n    after_margin = scores[1, target] - np.max(np.delete(scores[1], target))\n    assert after_margin > base_margin, "edge coupling failed directional self-test"\n    assert int(np.argmax(scores[1])) == target, "edge coupling selected wrong state"\n    # Confusion remap must be evidence-geometry preserving.\n    z = np.array([[3., 2., 0.], [0., 3., 2.], [2., 0., 3.], [2., 3., 0.]])\n    y = np.array([0, 0, 2, 0])\n    C = np.array([[.5, .1, .4], [.2, .6, .2], [.3, .2, .5]])\n    zz = impose_confusion_identity(z, y, C, 123)\n    assert np.array_equal(np.argmax(z,1)==y, np.argmax(zz,1)==y)\n    ks = np.array([5, 7, 11]); m = topk_mask(scores, ks)\n    assert np.all(m.sum(axis=1) == ks), "candidate capacity invariant failed"\n    print("SELF TEST: PASS")\n\n\ndef paired_wilcoxon(x: List[float]) -> Dict:\n    x = np.asarray(x, float)\n    nonzero = x[np.abs(x) > 1e-12]\n    if len(nonzero) == 0:\n        p = 1.0\n    else:\n        try: p = float(wilcoxon(x, alternative="two-sided", zero_method="wilcox").pvalue)\n        except Exception: p = float(\'nan\')\n    return {"mean": float(x.mean()), "positive": int((x > 0).sum()), "n": int(len(x)), "p": p}\n\n\ndef run_seed(seed: int, cfg: Config, outdir: Path):\n    train_df = make_train_dataframe(seed, cfg)\n    calib_df, calib_edges, calib_edges_df = make_calibration_dataframe(seed, cfg)\n    test_df, edge_tables, edges_df = make_chain_dataframe(seed, cfg, split="test", tag="test")\n\n    if cfg.semantic_backend == "pretrained":\n        tokenizer, model = train_pretrained(seed, train_df, cfg)\n        sem_cal = pretrained_logits(tokenizer, model, calib_df)\n        sem = pretrained_logits(tokenizer, model, test_df)\n    elif cfg.semantic_backend == "tiny":\n        model = train_transformer(seed, train_df, cfg)\n        sem_cal = transformer_logits(model, calib_df)\n        sem = transformer_logits(model, test_df)\n    else:\n        raise ValueError(f"unknown semantic_backend={cfg.semantic_backend}")\n\n    syn_cal, syn_acc, signals, confusions_cal, confusions_test = make_accuracy_controls(seed, calib_df, test_df, sem_cal, sem)\n    ys_cal = {"ref": calib_df.ref.to_numpy(int), "fam": calib_df.fam.to_numpy(int), "dir": calib_df.dir.to_numpy(int)}\n    ys_test = {"ref": test_df.ref.to_numpy(int), "fam": test_df.fam.to_numpy(int), "dir": test_df.dir.to_numpy(int)}\n\n    temps, syn_sharp, syn_conf = {}, {}, {}\n    for h in ("ref", "fam", "dir"):\n        # Fit on independent calibration data, then freeze for test.\n        T = fit_temperature(syn_cal[h], sem_cal[h], ys_cal[h])\n        temps[h] = T\n        syn_sharp[h] = syn_acc[h] / T\n        syn_conf[h] = impose_confusion_identity(\n            syn_sharp[h], ys_test[h], confusions_test[h], seed + {"ref":51000,"fam":52000,"dir":53000}[h]\n        )\n\n    chains = test_df.chain_id.to_numpy()\n    ystate = test_df.state_id.to_numpy(int)\n    shuffled_edges = shuffled_edge_tables(edge_tables, seed)\n\n    arms = (\n        ("synthetic_accuracy_only", syn_acc),\n        ("synthetic_sharpness_matched", syn_sharp),\n        ("synthetic_confusion_matched", syn_conf),\n        ("semantic_raw", sem),\n    )\n    arm_data = {}\n    row = {"seed": seed}\n    for h in ("ref", "fam", "dir"):\n        row[f"temperature_{h}"] = temps[h]\n        for split_name in ("calibration","test"):\n            for cls, sig in enumerate(signals[h][split_name]): row[f"signal_{split_name}_{h}_{cls}"] = float(sig)\n        for cname, C in (("calib", confusions_cal[h]), ("test", confusions_test[h])):\n            for i in range(C.shape[0]):\n                for j in range(C.shape[1]): row[f"{cname}_confusion_{h}_{i}_{j}"] = float(C[i,j])\n\n    for arm, logits in arms:\n        E = joint_log_emission(logits)\n        P = forward_backward(E, edge_tables, chains)\n        Ps = forward_backward(E, shuffled_edges, chains)\n        local = exact_state_accuracy(E, ystate); coupled = exact_state_accuracy(P, ystate); shuf = exact_state_accuracy(Ps, ystate)\n        row[f"{arm}_local_acc"] = local\n        row[f"{arm}_coupled_acc"] = coupled\n        row[f"{arm}_shuffled_acc"] = shuf\n        row[f"{arm}_delta"] = coupled - local\n        row[f"{arm}_coupling_minus_shuffled"] = coupled - shuf\n        for k,v in factor_metrics(logits, test_df).items(): row[f"{arm}_{k}"] = v\n        serial = serial_error_against_independent_null(\n            logits, test_df, edges_df, seed + {\n                "synthetic_accuracy_only":61000, "synthetic_sharpness_matched":62000,\n                "synthetic_confusion_matched":63000, "semantic_raw":64000\n            }[arm], simulations=cfg.null_simulations\n        )\n        for k,v in serial.items(): row[f"{arm}_{k}"] = v\n        arm_data[arm] = (E, P)\n\n    # Candidate diagnostic on the accuracy-only Gaussian arm for continuity.\n    E, P = arm_data["synthetic_accuracy_only"]\n    rank_scores = one_pass_coupling_scores(E, edge_tables, chains)\n    shuf_scores = one_pass_coupling_scores(E, shuffled_edges, chains)\n    diag = candidate_diagnostic(E, P, rank_scores, shuf_scores, test_df, edge_tables, cfg)\n    row.update({f"diag_{k}": v for k,v in diag.items()})\n\n    pos = test_df[["seed","chain_id","discourse_id","position","clause_id","state_id","ref","fam","dir","scenario","context_text","entity_0","entity_1","entity_2","entity_3","raw_text","text"]].copy()\n    for arm, (E,P) in arm_data.items():\n        pos[f"{arm}_local_pred"] = np.argmax(E, axis=1)\n        pos[f"{arm}_coupled_pred"] = np.argmax(P, axis=1)\n        pos[f"{arm}_local_correct"] = (pos[f"{arm}_local_pred"].to_numpy() == ystate).astype(int)\n        pos[f"{arm}_coupled_correct"] = (pos[f"{arm}_coupled_pred"].to_numpy() == ystate).astype(int)\n    pos.to_csv(outdir / f"seed{seed}_positions.csv", index=False)\n    test_df[["seed","chain_id","discourse_id","position","clause_id","scenario","context_text","entity_0","entity_1","entity_2","entity_3","raw_text","text","ref","fam","dir","state_id"]].to_csv(outdir / f"seed{seed}_clauses.csv", index=False)\n    edges_df.to_csv(outdir / f"seed{seed}_edges.csv", index=False)\n    calib_edges_df.to_csv(outdir / f"seed{seed}_calibration_edges.csv", index=False)\n    with open(outdir / f"seed{seed}_confusions.json", "w") as f:\n        json.dump({"calibration": {h: confusions_cal[h].tolist() for h in confusions_cal},\n                   "test": {h: confusions_test[h].tolist() for h in confusions_test}}, f, indent=2)\n    return row, test_df, edges_df\n\n\ndef make_report(summary: pd.DataFrame, cfg: Config) -> str:\n    def weighted_metric(value_col: str) -> float:\n        vals = summary[value_col].to_numpy(float); w = summary["diag_rescued_n"].to_numpy(float)\n        ok = np.isfinite(vals) & (w > 0)\n        return float(np.average(vals[ok], weights=w[ok])) if ok.any() else float("nan")\n\n    lines = [\n        "# Adrianic v5.4c Persistent-Discourse + Serial-Null Report", "",\n        f"Semantic backend: {cfg.semantic_backend}",\n        f"Model: {cfg.pretrained_model if cfg.semantic_backend == \'pretrained\' else \'v5.1 tiny transformer control\'}",\n        f"Seeds: {cfg.seeds}", f"Clauses/seed: {cfg.chains_per_seed*cfg.chain_len}",\n        f"Total clauses: {cfg.seeds*cfg.chains_per_seed*cfg.chain_len}",\n        f"Edge reliability ref/fam/dir: {cfg.edge_ref_reliability:.3f}/{cfg.edge_fam_reliability:.3f}/{cfg.edge_dir_reliability:.3f}",\n        "Every adjacency uses a freshly sampled ref permutation, fam permutation, and dir XOR; inference sees the exact edge constraint.",\n        "Within each chain, the same four entity names, scenario, and context sentence persist; only referring expression and relation template vary.",\n        f"Independent-error null simulations per arm/seed: {cfg.null_simulations}", ""\n    ]\n    arms = ("synthetic_accuracy_only","synthetic_sharpness_matched","synthetic_confusion_matched","semantic_raw")\n    for arm in arms:\n        loc=summary[f"{arm}_local_acc"].mean(); coup=summary[f"{arm}_coupled_acc"].mean(); sh=summary[f"{arm}_shuffled_acc"].mean()\n        stat=paired_wilcoxon(summary[f"{arm}_delta"].tolist()); nullstat=paired_wilcoxon(summary[f"{arm}_coupling_minus_shuffled"].tolist())\n        lines += [f"## {arm}", f"local/coupled/shuffled: {loc:.4f}/{coup:.4f}/{sh:.4f}",\n                  f"coupled-local: {100*(coup-loc):+.3f} pp; {stat[\'positive\']}/{stat[\'n\']} positive, p={stat[\'p\']:.4g}",\n                  f"coupled-shuffled-edge: {100*(coup-sh):+.3f} pp; {nullstat[\'positive\']}/{nullstat[\'n\']} positive, p={nullstat[\'p\']:.4g}", ""]\n\n    lines += ["## Emission-control audit"]\n    for h in ("ref","fam","dir"):\n        a=summary[f"synthetic_accuracy_only_{h}_acc"].mean(); b=summary[f"synthetic_sharpness_matched_{h}_acc"].mean(); c=summary[f"synthetic_confusion_matched_{h}_acc"].mean(); s=summary[f"semantic_raw_{h}_acc"].mean()\n        sa=summary[f"synthetic_accuracy_only_{h}_strength"].mean(); sb=summary[f"synthetic_sharpness_matched_{h}_strength"].mean(); sc=summary[f"synthetic_confusion_matched_{h}_strength"].mean(); ss=summary[f"semantic_raw_{h}_strength"].mean()\n        lines.append(f"{h}: acc A/S/C/semantic={a:.3f}/{b:.3f}/{c:.3f}/{s:.3f}; strength={sa:.3f}/{sb:.3f}/{sc:.3f}/{ss:.3f}; calibration T={summary[f\'temperature_{h}\'].mean():.3f}")\n    lines += ["", "The evaluated Gaussian controls are marginal-accuracy locked to semantic per-class accuracy on the current persistent-context corpus. Confusion matching changes only wrong-class identity: row correctness, true logit, true margin, centered strength, and sorted logit spectrum are invariant by assertion.", ""]\n\n    lines += ["## Serial error dependence vs empirical independent-error null"]\n    for arm in ("synthetic_sharpness_matched","synthetic_confusion_matched","semantic_raw"):\n        lines.append(f"### {arm}")\n        for h in ("ref","fam","dir"):\n            lag=summary[f"{arm}_{h}_error_lag1_corr"].mean(); lnull=summary[f"{arm}_{h}_error_lag1_corr_null_mean"].mean(); lex=summary[f"{arm}_{h}_error_lag1_corr_excess"].mean(); lz=summary[f"{arm}_{h}_error_lag1_corr_z"].mean()\n            vr=summary[f"{arm}_{h}_chain_error_rate_var"].mean(); vnull=summary[f"{arm}_{h}_chain_error_rate_var_null_mean"].mean(); vex=summary[f"{arm}_{h}_chain_error_rate_var_excess"].mean()\n            lines.append(f"{h}: lag1 obs/null/excess={lag:+.4f}/{lnull:+.4f}/{lex:+.4f} (mean z={lz:+.2f}); chain-error variance obs/null/excess={vr:.5f}/{vnull:.5f}/{vex:+.5f}")\n            if h in ("ref","fam"):\n                coh=summary[f"{arm}_{h}_both_wrong_edge_coherence"].mean(); cnull=summary[f"{arm}_{h}_both_wrong_edge_coherence_null_mean"].mean(); cex=summary[f"{arm}_{h}_both_wrong_edge_coherence_excess"].mean(); cz=summary[f"{arm}_{h}_both_wrong_edge_coherence_z"].mean(); n=summary[f"{arm}_{h}_both_wrong_edges"].sum()\n                lines.append(f"  both-wrong edge coherence obs/null/excess={coh:.4f}/{cnull:.4f}/{cex:+.4f} (mean z={cz:+.2f}, observed edges={int(n)})")\n        lines.append("direction edge-coherence intentionally omitted: binary XOR makes it non-diagnostic when conditioning on both wrong.")\n    lines.append("")\n\n    cov_local=summary.diag_local_coverage.mean(); cov_c=summary.diag_coupling_coverage.mean(); cov_s=summary.diag_shuffled_coverage.mean()\n    stat_l=paired_wilcoxon(summary.diag_coupling_minus_local_pp.tolist()); stat_s=paired_wilcoxon(summary.diag_coupling_minus_shuffled_pp.tolist())\n    lines += ["## Candidate diagnostic", f"mean candidate budget: {summary.diag_mean_k.mean():.3f}",\n              f"coverage local/coupling/shuffled-edge: {cov_local:.4f}/{cov_c:.4f}/{cov_s:.4f}",\n              f"coupling-local: {summary.diag_coupling_minus_local_pp.mean():+.3f} pp; {stat_l[\'positive\']}/{stat_l[\'n\']} positive, p={stat_l[\'p\']:.4g}",\n              f"coupling-shuffled: {summary.diag_coupling_minus_shuffled_pp.mean():+.3f} pp; {stat_s[\'positive\']}/{stat_s[\'n\']} positive, p={stat_s[\'p\']:.4g}",\n              f"rescued clauses: {int(summary.diag_rescued_n.sum())}",\n              f"rescued local argmax: {weighted_metric(\'diag_rescued_local_argmax\'):.4f}",\n              f"rescued coupling softwall: {weighted_metric(\'diag_rescued_coupling_softwall\'):.4f}",\n              f"rescued structured exact: {weighted_metric(\'diag_rescued_structured_exact\'):.4f}",\n              f"rescued exact candidate-conditioned: {weighted_metric(\'diag_rescued_structured_exact_candidate_conditioned\'):.4f}", ""]\n    lines += ["## Interpretation ladder",\n              "A → S isolates sharpness after accuracy calibration.",\n              "S → C isolates marginal confusion identity while preserving every row\'s evidence geometry.",\n              "C → semantic is the residual after matching one-site marginals; persistent discourse gives that residual a legitimate serial channel.",\n              "Raw serial statistics are never compared to naive chance. They are compared with independent predictions sampled from the arm\'s own one-site confusion matrix on the exact same chains.",\n              "A positive null-adjusted lag/chain-variance signal indicates dependence beyond one-site confusion and is the primary semantic diagnostic.",\n              "Ref/fam edge-coherence is retained only as a negative-control/leakage check: persistent discourse does not reveal the fresh random edge permutation to the clause model. Direction edge-coherence is omitted entirely."]\n    return "\\n".join(lines) + "\\n"\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--seeds", type=int, default=8)\n    p.add_argument("--chains", type=int, default=8)\n    p.add_argument("--chain-len", type=int, default=80)\n    p.add_argument("--train-repeats", type=int, default=14)\n    p.add_argument("--epochs", type=int, default=8, help="tiny backend epochs")\n    p.add_argument("--backend", choices=["tiny","pretrained"], default="tiny")\n    p.add_argument("--model", type=str, default="distilbert-base-uncased")\n    p.add_argument("--pretrained-epochs", type=int, default=4)\n    p.add_argument("--null-sims", type=int, default=500)\n    p.add_argument("--out", type=str, default="/mnt/data/adrianic_v54c_persistent_discourse_results")\n    p.add_argument("--smoke", action="store_true")\n    args = p.parse_args()\n    cfg = Config(seeds=args.seeds, chains_per_seed=args.chains, chain_len=args.chain_len, train_repeats_per_state=args.train_repeats, transformer_epochs=args.epochs, semantic_backend=args.backend, pretrained_model=args.model, pretrained_epochs=args.pretrained_epochs, null_simulations=args.null_sims, output_dir=args.out)\n    if args.smoke:\n        cfg.seeds=min(cfg.seeds,2); cfg.chains_per_seed=min(cfg.chains_per_seed,2); cfg.chain_len=min(cfg.chain_len,40); cfg.train_repeats_per_state=min(cfg.train_repeats_per_state,6); cfg.transformer_epochs=min(cfg.transformer_epochs,3)\n    outdir = Path(cfg.output_dir); outdir.mkdir(parents=True, exist_ok=True)\n    (outdir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))\n    self_test()\n    rows=[]; all_clauses=[]; all_edges=[]\n    for seed in range(cfg.seeds):\n        print(f"seed {seed+1}/{cfg.seeds}", flush=True)\n        row, clauses, edges = run_seed(seed, cfg, outdir)\n        rows.append(row); all_clauses.append(clauses); all_edges.append(edges)\n        print(f"  A Δ={100*row[\'synthetic_accuracy_only_delta\']:+.2f}pp S Δ={100*row[\'synthetic_sharpness_matched_delta\']:+.2f}pp C Δ={100*row[\'synthetic_confusion_matched_delta\']:+.2f}pp semantic Δ={100*row[\'semantic_raw_delta\']:+.2f}pp rescued={row[\'diag_rescued_n\']}", flush=True)\n    summary=pd.DataFrame(rows); summary.to_csv(outdir/"summary.csv", index=False)\n    pd.concat(all_clauses, ignore_index=True).to_csv(outdir/"clauses.csv", index=False)\n    pd.concat(all_edges, ignore_index=True).to_csv(outdir/"edges.csv", index=False)\n    report=make_report(summary,cfg); (outdir/"REPORT.md").write_text(report); print("\\n"+report)\n\n\nif __name__ == "__main__":\n    main()\n'
SCRIPT = Path("/content/Adrianic_v5_4c_PersistentDiscourse_SerialNull.py")
SCRIPT.write_text(script)
print("wrote", SCRIPT, "bytes", SCRIPT.stat().st_size)


In [ ]:
# CELL 3 — Run the decisive pretrained semantic experiment
OUT = "/content/adrianic_v54c_persistent_discourse_results"
!python /content/Adrianic_v5_4c_PersistentDiscourse_SerialNull.py \
  --backend pretrained \
  --model distilbert-base-uncased \
  --seeds 8 --chains 8 --chain-len 80 \
  --train-repeats 14 --pretrained-epochs 4 \
  --null-sims 500 --out {OUT}


In [ ]:
# CELL 4 — Inspect the report and package every result
from pathlib import Path
from IPython.display import Markdown, display
report = Path(OUT) / "REPORT.md"
display(Markdown(report.read_text()))
!cd /content && zip -qr Adrianic_v5_4c_RESULTS.zip adrianic_v54c_persistent_discourse_results
print("Result zip: /content/Adrianic_v5_4c_RESULTS.zip")


## Readout rule

The primary empirical quantities are the semantic arm’s **null-adjusted lag-1 error correlation** and **null-adjusted within-chain error-rate variance**, together with its coupled-minus-local accuracy delta. Ref/family edge-coherence is only a negative-control/leakage check; direction edge-coherence is not evaluated.
